# Book Rating Prediction

This notebook builds a machine learning pipeline to predict a book's average rating from Goodreads-style metadata. We load and clean the data, engineer features, train and compare four regression models, look at feature importance, and save the final model for use in a Streamlit app.

## Loading the Data

The raw CSV has 11,128 lines. Some rows don't match the expected number of columns (usually caused by commas or quotes inside book titles), so we use `on_bad_lines='skip'` to skip them while loading. This leaves about 11,123 rows.

In [15]:
import pandas as pd

# on_bad_lines='skip' = don't crash on rows with the wrong number of fields
df = pd.read_csv("books.csv", on_bad_lines="skip")

print("Shape (rows, cols):", df.shape)
df.head()

Shape (rows, cols): (11123, 12)


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


We used `.info()`, `.describe()` and null counts to check the data types, value ranges and missing values before doing any cleaning.

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  str    
 2   authors             11123 non-null  str    
 3   average_rating      11123 non-null  float64
 4   isbn                11123 non-null  str    
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  str    
 7     num_pages         11123 non-null  int64  
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  str    
 11  publisher           11123 non-null  str    
dtypes: float64(1), int64(5), str(6)
memory usage: 2.1 MB


In [17]:
df.describe()

,bookID,average_rating,isbn13,num_pages,ratings_count,text_reviews_count
count,11123.000000,11123.000000,1.112300e+04,11123.000000,1.112300e+04,11123.000000
mean,21310.856963,3.934075,9.759880e+12,336.405556,1.794285e+04,542.048099
std,13094.727252,0.350485,4.429758e+11,241.152626,1.124992e+05,2576.619589
min,1.000000,0.000000,8.987060e+09,0.000000,0.000000e+00,0.000000
25%,10277.500000,3.770000,9.780345e+12,192.000000,1.040000e+02,9.000000
50%,20287.000000,3.960000,9.780582e+12,299.000000,7.450000e+02,47.000000
75%,32104.500000,4.140000,9.780872e+12,416.000000,5.000500e+03,238.000000
max,45641.000000,5.000000,9.790008e+12,6576.000000,4.597666e+06,94265.000000


In [18]:
print("Null counts:")
print(df.isnull().sum())

print("\nColumn names (repr shows hidden spaces):")
for col in df.columns:
    print(repr(col))

Null counts:
bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
  num_pages           0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

Column names (repr shows hidden spaces):
'bookID'
'title'
'authors'
'average_rating'
'isbn'
'isbn13'
'language_code'
'  num_pages'
'ratings_count'
'text_reviews_count'
'publication_date'
'publisher'


### Initial observations

- The dataset has 11,123 rows and 12 columns after loading.
- There are no NaN values, but missing data is represented as 0 instead. For example, `average_rating` and `num_pages` both have a minimum of 0.
- The `num_pages` column name has a leading space, which needs to be fixed before it can be used.
- `publication_date` is still stored as text, not as a real date.
- `bookID`, `isbn` and `isbn13` are identifiers and won't be used as features.
- The target column is `average_rating`.


## Data Cleaning

Four things need to be fixed before the data is ready for modelling: the `num_pages` column name has extra spaces, some `publication_date` values are invalid, `average_rating == 0` actually means the book has no rating, and `num_pages == 0` means the page count is missing.

### Fixing Column Names

The `num_pages` column has leading spaces in its name, so we strip whitespace from all column names.

In [19]:
df.columns = df.columns.str.strip()
print([repr(c) for c in df.columns])

["'bookID'", "'title'", "'authors'", "'average_rating'", "'isbn'", "'isbn13'", "'language_code'", "'num_pages'", "'ratings_count'", "'text_reviews_count'", "'publication_date'", "'publisher'"]


### Invalid Publication Dates

A few rows have impossible dates, such as 11/31/2000 (November only has 30 days). We parse the dates with `errors='coerce'`, which turns invalid dates into NaT, and drop those rows. Only 2 rows are affected.

In [20]:
df["publication_date_parsed"] = pd.to_datetime(
    df["publication_date"], format="%m/%d/%Y", errors="coerce"
)

bad_dates = df["publication_date_parsed"].isna()
print("Invalid publication_date rows:", bad_dates.sum())
print(df.loc[bad_dates, ["title", "publication_date"]].head())

df = df.loc[~bad_dates].copy()
print("Shape after dropping bad dates:", df.shape)

Invalid publication_date rows: 2
                                                   title publication_date
8177   In Pursuit of the Proper Sinner (Inspector Lyn...       11/31/2000
11094         Montaillou  village occitan de 1294 à 1324        6/31/1982
Shape after dropping bad dates: (11121, 13)


### Dropping Rows with average_rating == 0

A rating of exactly 0 doesn't really happen once a book has actual ratings, so we treat it as a missing label rather than a real score and drop these rows. 25 rows are affected.

In [21]:
print("Rows with average_rating == 0:", (df["average_rating"] == 0).sum())
df = df.loc[df["average_rating"] != 0].copy()
print("Shape after dropping zero ratings:", df.shape)

Rows with average_rating == 0: 25
Shape after dropping zero ratings: (11096, 13)


### Handling Missing Page Counts

About 76 books have `num_pages == 0`. Instead of dropping these rows, since they still have a valid rating and other useful features, we impute the missing page count with the median and add a `pages_missing_flag` column so the model can tell which rows were imputed.

In [22]:
df["pages_missing_flag"] = (df["num_pages"] == 0).astype(int)

median_pages = df.loc[df["num_pages"] > 0, "num_pages"].median()
print("Median pages (nonzero only):", median_pages)
print("Rows imputed:", df["pages_missing_flag"].sum())

df.loc[df["num_pages"] == 0, "num_pages"] = median_pages

print("num_pages == 0 remaining:", (df["num_pages"] == 0).sum())
print("Final shape after Step 2:", df.shape)
df[["num_pages", "pages_missing_flag", "average_rating"]].head()

Median pages (nonzero only): 302.0
Rows imputed: 76
num_pages == 0 remaining: 0
Final shape after Step 2: (11096, 14)


,num_pages,pages_missing_flag,average_rating
0,652,0,4.57
1,870,0,4.49
2,352,0,4.42
3,435,0,4.56
4,2690,0,4.78


### Summary of Cleaning Steps

| Action | Rule applied |
|---|---|
| Column names | stripped whitespace |
| Invalid dates | parsed, then dropped (~2 rows) |
| average_rating == 0 | dropped (~25 rows) |
| num_pages == 0 | median imputed + flagged (~76 rows) |

We treat these two zero-cases differently because `average_rating` is the target — we can't invent the label — while `num_pages` is just one of several features, so imputing it is reasonable.